# Import

In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
import h5py
%load_ext autoreload
%autoreload 2

import mapd.sentinels  # load once
%aimport -mapd.sentinels

%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)

RECOMPUTE = True

# Create Figure 4 sinq - MN spiking

In [ ]:
sinq = Sinq(sinqname='Figure4')
print(sinq.__repr__())

sinq.display_df()

Found data directory: D:\Data
Sinq(Figure4, 2 rows x 30 params): 2 empties; ['210302_F1_C2', '210319_F2_C1'] x ['Table', 'parquet', 'genotype', 'notes', 'duration', 'rms_velocity', 'outcome_fractions_no_as_no_mv', 'outcome_fractions_as_off', 'outcome_fractions_rest', 'outcome_fractions_no_as_mv', 'outcome_fractions_probe', 'outcome_fractions_as_off_late', 'outcome_fractions_timeout_fail', 'outcome_fractions_timeout', 'outcome_fractions_info', 'lo_state_median_position', 'hi_state_median_position', 'hi_lo_shift', 'hi_state_on_target', 'lo_state_on_target', 'num_trials', 'blue_fraction', 'blue_toggle_fraction', 'most_common_fiberLED', 'lo_target_off_state', 'hi_target_off_state', 'probe_positive_effort', 'successes', 'hard_successes', 'holding_cost'] 


,Table,parquet,genotype,notes,duration,rms_velocity,outcome_fractions_no_as_no_mv,outcome_fractions_as_off,outcome_fractions_rest,outcome_fractions_no_as_mv,...,num_trials,blue_fraction,blue_toggle_fraction,most_common_fiberLED,lo_target_off_state,hi_target_off_state,probe_positive_effort,successes,hard_successes,holding_cost
210302_F1_C2,None,LEDFlashWithPiezoCueControl_210302_F1_C2_Table...,Hot-Cell-Gal4 (test),None,6836.72718,32982.968041,0.561250,0.173750,0.118750,0.111250,...,800.0,1.0,0.000000,epi_only,0.055824,0.082934,8.684294e+05,78.0,37.0,9.147350e+06
210319_F2_C1,None,LEDFlashWithPiezoCueControl_210319_F2_C1_Table...,Hot-Cell-LexA_Chr;81A06_pJFRC7,None,6141.59006,63494.653833,0.236697,0.231193,0.119266,0.111927,...,545.0,1.0,0.239583,epi_only,0.049627,0.018823,1.655564e+06,58.0,18.0,5.410378e+06


In [ ]:
# Standard compute
def run_compute(sinq,id):
    if id in sinq.df.index:
        T = sinq.restore_table(dayflycell=id,overwrite=True)
        # endofdaytrials = T.df.loc[1081:]
        # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
    else:
        T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
        sinq.add_table(table=T)
        sinq.df
    T.exclude_trials()
    T.extract_trial_properties()
    T.find_successful_trials()
    return sinq,T


# Standard plotting
def run_plots(T):
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

    df_filters = {'all':None,
              'hi':{'pyasState':'hi'},
              'lo':{'pyasState':'lo'},}

    for filt_key in df_filters.keys():
        dff = df_filters[filt_key]
        fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                                df_filter=dff,index=None,savefig=False,format='png',
                                export=True)

# Whole cell recordings

### 210302_F1_C2

In [ ]:
id = '210302_F1_C2'
if not sinq.df is None and id in sinq.df.index:
    T = sinq.restore_table(dayflycell=id,overwrite=True)
    # endofdaytrials = T.df.loc[1081:]
    # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
else:
    T = Table(f'LEDFlashWithPiezoCueControl_{id}_Table.parquet')
    T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
    T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
    sinq.add_table(table=T)
    sinq.df
T.exclude_trials()
T.extract_trial_properties()
T.find_successful_trials()
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

In [ ]:
id = '210302_F1_C2'
# sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= 'premn',
        tags=['nonmotor','random'],
        append=False,
)

if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)

### 210319_F2_C1 - Do not use
What is the deal!? There is an enormous number of points and the distributions is near probe_Zero, but the heatmap shows displacement. Not clear where this is coming from...?

In [ ]:
# T = Table('LEDFlashWithPiezoCueControl_210319_F2_C1_Table.parquet',progress_bar=True,add_probestate=False)
# # Found 4 target positions - Counter({('lo', 730.0, -180.0, 60.0): 251, ('hi', 730.0, -260.0, 60.0): 199, ('no_state', 0.0, 520.0, 80.0): 50, ('no_state', 0.0, 450.0, 80.0): 50})
# # T.df.loc[T.df['pyasXPosition']==490,'Trial'].index
# # T.exclude_list_of_trials(T.df.loc[T.df['pyasXPosition']==490,'Trial'].index,reason='unused target position')
# # T.df.loc[T.df['pyasXPosition']==520,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==450,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==520,'pyasState'] = 'lo'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState'] = 'hi'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState']
# # T.write_column_to_trial_files('pyasState')
# # T.write_column_to_trial_files('probeZero')
# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# trial = Trial('LEDFlashWithPiezoCueControl_Raw_210319_F2_C1_1.mat')
# trial.fiberLED

In [ ]:
id = '210319_F2_C1'
if not sinq.df is None and id in sinq.df.index:
    T = sinq.restore_table(dayflycell=id,overwrite=True)
    # endofdaytrials = T.df.loc[1081:]
    # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
else:
    T = Table(f'LEDFlashWithPiezoCueControl_{id}_Table.parquet')
    T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
    T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
    sinq.add_table(table=T)
    sinq.df
T.exclude_trials()
T.extract_trial_properties()
T.find_successful_trials()
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

In [ ]:
id = '210319_F2_C1'
# sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= '81A06',
        tags=['','F?'],
        append=False,
)

if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)

if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)

### 210331_F2_C1 - HC-LexA/13XLexAop-ChrimsonR;81A06/pJFRC7 100uL ATR mixed in

In [ ]:
id = '210319_F2_C1'
# sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= '81A06',
        tags=['','F?'],
        append=False,
)

if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)

if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)

In [ ]:
id = '210402_F2_C1'
sinq.set_note(
        dayflycell = id,
        text= '81A06',
        tags=['','F?'],
        append=False,
)

In [ ]:
id = '210405_F1_C1'
sinq.set_note(
        dayflycell = id,
        text= '81A06',
        tags=['','F?'],
        append=False,
)

In [ ]:
id = '210602_F1_C1'
sinq.set_note(
        dayflycell = id,
        text= '35C09',
        tags=['','F?'],
        append=False,
)

In [ ]:
id = '210604_F1_C1'

In [ ]:
id = '210908_F3_C1'

In [ ]:
id = '210915_F1_C1'

In [ ]:
id = '210917_F2_C1'

In [ ]:
id = '240430_F1_C1'

In [ ]:
id = '241115_F1_C1'

In [ ]:
id = '241203_F2_C1'